# Used Car Price Prediction — PakWheels (Pakistan)

Machine learning pipeline for predicting used-car prices in Pakistan, built on ~2,500 listings
scraped from PakWheels covering Lahore, Karachi, and Islamabad.

**Pipeline:**
1. Data collection (Selenium-based scraper, Cloudflare-protected site)
2. Data cleaning and feature extraction
3. Exploratory data analysis
4. Company/model parsing from listing titles
5. Decision Tree Regressor for price prediction + saleability scoring
6. Model evaluation (overall and price-segment-wise)
7. KNN-based estimation of missing listing attributes

Hyperparameter tuning and final model selection are covered in a separate notebook
(`hyperparameter_tuning.ipynb`); this notebook uses a fixed baseline configuration.


## 1. Data Collection

PakWheels sits behind Cloudflare, so plain `requests` calls return `403`. The scraper below
uses Selenium (a real, controlled browser) to render each search page, then parses the DOM
with BeautifulSoup. It scrapes 25 pages each for Lahore, Karachi, and Islamabad, saves
incrementally to SQLite after every page, and auto-restarts the browser if it crashes —
so a failure mid-run doesn't cost the whole scrape.


In [ ]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time
import pandas as pd
import sqlite3

print("Initializing Fault-Tolerant Data Mining Engine...")

options = webdriver.EdgeOptions()
options.add_experimental_option("excludeSwitches", ["enable-logging"]) 
driver = webdriver.Edge(options=options)

cities = ['lahore', 'karachi', 'islamabad']
pages_per_city = 25

# 1. Open Database Connection
conn = sqlite3.connect('pakwheels_dataset.db')
cursor = conn.cursor() # direct sql execution

# Clear the old production table so we don't get duplicate data if you restart the script
cursor.execute("DROP TABLE IF EXISTS used_cars_production")
conn.commit()

try: # overall try block to catch any unexpected crashes and ensure we close the browser and database connection gracefully
    for city in cities:
        print(f"\n=== TARGETING CITY: {city.upper()} ===")
        for page in range(1, pages_per_city + 1):
            page_data = [] # Reset memory for each page
            url = f"https://www.pakwheels.com/used-cars/search/-/ct_{city}/?page={page}"
            print(f"[{city.upper()}] Extracting Page {page}/{pages_per_city}...")
            
            try: # try block for each page to handle potential crashes without losing the entire dataset
                driver.get(url)
                time.sleep(8)
                
                soup = BeautifulSoup(driver.page_source, 'html.parser')
                listings = soup.select('.classified-listing')
                
                for listing in listings:
                    title_node = listing.select_one('h3')
                    title = title_node.text.strip() if title_node else "N/A"
                    
                    price_node = listing.select_one('.price-details, .generic-green')
                    price = price_node.text.strip() if price_node else "N/A"
                    
                    specs_ul = listing.select_one('.search-vehicle-info-2')
                    if specs_ul:
                        specs = [li.text.strip() for li in specs_ul.find_all('li')]
                    else:
                        specs = ["N/A"]
                    
                    page_data.append({
                        "City": city.capitalize(),
                        "Title": title,
                        "Price_Raw": price,
                        "Specs_Raw": str(specs) #list goes in as string
                    })
                
                # 2. INCREMENTAL LOAD: Save to database immediately
                if page_data:
                    df = pd.DataFrame(page_data)
                    # 'append' ensures we add to the table without overwriting previous pages
                    df.to_sql('used_cars_production', conn, if_exists='append', index=False)
                    
            except Exception as e:
                # 3. AUTO-RECOVERY: Reboot the engine if the browser crashes
                print(f"-> Crash detected on {city} Page {page}: {e}")
                print("-> Attempting WebDriver engine reboot...")
                try:
                    driver.quit()
                except:
                    pass
                driver = webdriver.Edge(options=options)
                time.sleep(3) # Give the new browser a moment to initialize

finally:
    try:
        driver.quit()
    except:
        pass
    conn.close()
    print("\n--- FAULT-TOLERANT EXTRACTION COMPLETE ---")

# Let's verify what made it into the database
conn = sqlite3.connect('pakwheels_dataset.db')
final_df = pd.read_sql_query("SELECT * FROM used_cars_production", conn)
print(f"Total Rows Safely Secured in Database: {len(final_df)}")
conn.close()

# 1. Import libraries.
# 2. Start Edge browser.
# 3. Define target cities and pages.
# 4. Open SQLite database.
# 5. Delete old production table.
# 6. For each city:
#       For each page:
#           Create page URL.
#           Open page in browser.
#           Wait 8 seconds.
#           Parse HTML.
#           Find car listings.
#           Extract title, price, specs.
#           Store records in page_data.
#           Convert page_data to DataFrame.
#           Append DataFrame to SQLite table.
#           If browser crashes:
#               close old browser
#               start new browser
#               continue scraping next page
# 7. Close browser and database.
# 8. Reopen database.
# 9. Read saved table.
# 10. Print total rows saved.
# 11. Close database.

## 2. Data Cleaning

Raw fields (`Price_Raw`, `Specs_Raw`) are text scraped straight from the page. This step:
- Converts price strings (e.g. `"PKR 25 lacs"`, `"1.2 crore"`) into integer PKR values
- Splits the specs list into `Year`, `Mileage_KM`, `Fuel_Type`, `Engine_CC`, `Transmission`
- Drops rows missing any core numeric field, and drops exact duplicates


In [ ]:
import pandas as pd
import sqlite3
import ast  #This converts a string that looks like a Python list back into an actual Python list
import re   # we use it to extract numeric values from the price strings

print("Initiating Data Transformation Engine...")

# 1. Load the raw production data
conn = sqlite3.connect('pakwheels_dataset.db')
df = pd.read_sql_query("SELECT * FROM used_cars_production", conn) # we could use cursor and execute a SELECT statement

# if there is a string for price, this function converts it into an integer
def clean_price(price_str):
    if pd.isna(price_str) or price_str == "N/A" or price_str == "Call for price": 
        return None
    price_str = str(price_str).replace(',', '')  #"PKR 1,250,000" == > "PKR 1250000"
    
    # Isolate the numeric portion using RegEx
    match = re.search(r"(\d+(\.\d+)?)", price_str) #extracts the numeric portion as a string
#     Find a number.
# The number may be an integer like 45
# Or it may be decimal like 45.5
    
    if match:
        val = float(match.group(1))
        # Apply standard Pakistani currency multipliers
        if "lac" in price_str.lower(): 
            return int(val * 100000)
        elif "crore" in price_str.lower(): 
            return int(val * 10000000)
        return int(val)
    return None

df['Price_PKR'] = df['Price_Raw'].apply(clean_price)

# 3. Array Splitting Logic
def extract_specs(row):
    try:
        # specs becomes the actual python list
        specs = ast.literal_eval(row['Specs_Raw'])
        
        year = int(specs[0]) if len(specs) > 0 and specs[0].isdigit() else None  #isdigit checks if only number returns bool value
        
        mileage = specs[1].replace(',', '').replace(' km', '') if len(specs) > 1 else None
        mileage = int(mileage) if mileage and mileage.isdigit() else None
        
        fuel = specs[2] if len(specs) > 2 else None
        
        cc = specs[3].replace(' cc', '') if len(specs) > 3 else None
        cc = int(cc) if cc and cc.isdigit() else None
        
        transmission = specs[4] if len(specs) > 4 else None
        
        return pd.Series([year, mileage, fuel, cc, transmission])
    except Exception as e:
        print(e)
        return pd.Series([None, None, None, None, None])

# Broadcast the split arrays across 5 new distinct columns
df[['Year', 'Mileage_KM', 'Fuel_Type', 'Engine_CC', 'Transmission']] = df.apply(extract_specs, axis=1) #applies the function row by row axis = 1

# 4. Dimensionality Reduction (Drop useless/corrupted rows)
df = df.drop(columns=['Price_Raw', 'Specs_Raw'])

#  works only on the three colums provided, if any of them is null, rntire row is dropped
df = df.dropna(subset=['Price_PKR', 'Year', 'Mileage_KM', 'Engine_CC'])
df = df.drop_duplicates(subset=['City', 'Title', 'Price_PKR', 'Year', 'Mileage_KM', 'Engine_CC'])

# they may be floats in pandas series
df['Price_PKR'] = df['Price_PKR'].astype(int)
df['Year'] = df['Year'].astype(int)
df['Mileage_KM'] = df['Mileage_KM'].astype(int)

print(f"\nTotal Usable Rows After Cleaning: {len(df)}")
display(df.head())

# 5. Commit Final Dataset
df.to_sql('cleaned_cars', conn, if_exists='replace', index=False)
conn.close()

print("\nSuccess: Cleaned, mathematically valid dataset saved to 'cleaned_cars' table.")

## 3. Exploratory Data Analysis

IEEE-style figures used in the report, saved to `report_feature_figures/`.


In [ ]:
# ============================================================
# Exploratory Data Analysis (EDA) - Initial Visualizations
# ============================================================

import os
import sqlite3

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import seaborn as sns
import matplotlib.patheffects as pe

print("Initializing EDA Engine...")

# ============================================================
# Load cleaned production dataset
# ============================================================

conn = sqlite3.connect("pakwheels_dataset.db")
df = pd.read_sql_query("SELECT * FROM cleaned_cars", conn)
conn.close()

# ============================================================
# Create folder for report figures
# ============================================================

os.makedirs("report_feature_figures", exist_ok=True)

# ============================================================
# Global plotting configuration (IEEE style)
# ============================================================

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.family": "DejaVu Sans",

    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    "axes.linewidth": 0.8,

    "axes.spines.top": False,
    "axes.spines.right": False,

    "legend.frameon": False
})

# ============================================================
# Academic colour palette
# ============================================================

COLORS = {
    "primary": "#355C7D",
    "secondary": "#6C5B7B",
    "accent": "#2A9D8F"
}

# ============================================================
# Helper Functions
# ============================================================

def format_millions(x, pos):
    """Format PKR values in millions."""
    return f"{x/1_000_000:.0f}M"

def style_axes(ax):
    """Apply consistent IEEE styling."""
    ax.grid(axis="y",
            linestyle="--",
            linewidth=0.6,
            alpha=0.45)

    ax.grid(axis="x", visible=False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(direction="out")

# ============================================================
# Figure 1 : Price Distribution
# ============================================================

fig, ax = plt.subplots(figsize=(8,5))

sns.histplot(
    df["Price_PKR"],
    bins=50,
    kde=True,
    color=COLORS["primary"],
    alpha=0.75,
    edgecolor="white",
    linewidth=0.4,
    line_kws={"linewidth":2.2},
    ax=ax
)

ax.set_xlabel("Price (PKR)")
ax.set_ylabel("Frequency")

ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(format_millions)
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/price_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# Figure 3 : Correlation Heatmap
# ============================================================

numeric_cols = df[
    [
        "Price_PKR",
        "Year",
        "Mileage_KM",
        "Engine_CC"
    ]
]

corr_matrix = numeric_cols.corr()

fig, ax = plt.subplots(figsize=(6.5,5.5))

sns.heatmap(
    corr_matrix,

    annot=True,

    fmt=".2f",

    cmap="coolwarm",

    square=True,

    linewidths=0.5,

    annot_kws={
        "size":11
    },

    cbar_kws={
        "shrink":0.9
    },

    ax=ax
)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("EDA visualizations generated successfully.")

In [ ]:
price = df['Price_PKR']  # Replace df with your DataFrame name

print("========== PRICE STATISTICS ==========")
print(f"Minimum Price : PKR {price.min():,.0f}")
print(f"Maximum Price : PKR {price.max():,.0f}")
print(f"Mean Price    : PKR {price.mean():,.0f}")
print(f"Median Price  : PKR {price.median():,.0f}")
print(f"Std Deviation : PKR {price.std():,.0f}")

print("\n========== PERCENTILES ==========")
print(price.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

In [ ]:
# ============================================================
# Additional Exploratory Visualizations
# (Notebook Analysis / Backup Figures)
# ============================================================

# ------------------------------------------------------------
# Figure 1 : Price Boxplot
# Purpose:
# Visualize the spread of vehicle prices, highlighting the
# median, mean, quartiles, and potential outliers.
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.5))

sns.boxplot(
    x=df["Price_PKR"],
    color=COLORS["primary"],
    width=0.45,
    linewidth=1.2,
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 6
    },
    ax=ax
)

ax.set_xlabel("Price (PKR)")
ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(format_millions)
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/price_boxplot.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# Figure 2 : Price Distribution (95% Display Range)
# Purpose:
# Display the complete dataset while zooming into the
# first 95% of the price range for better readability.
# ------------------------------------------------------------

upper_price = df["Price_PKR"].quantile(0.95)

fig, ax = plt.subplots(figsize=(8,5))

sns.histplot(
    df["Price_PKR"],
    bins=40,
    kde=True,
    color=COLORS["primary"],
    alpha=0.75,
    edgecolor="white",
    linewidth=0.4,
    line_kws={"linewidth":2},
    ax=ax
)

ax.set_xlim(0, upper_price)

ax.set_xlabel("Price (PKR)")
ax.set_ylabel("Frequency")

ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(format_millions)
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/price_distribution_zoomed.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# Figure 3 : Price Distribution (Top 5% Removed)
# Purpose:
# Show the central distribution after removing the
# highest-priced 5% of observations.
# ------------------------------------------------------------

filtered_price_df = df[
    df["Price_PKR"] <= df["Price_PKR"].quantile(0.95)
]

fig, ax = plt.subplots(figsize=(8,5))

sns.histplot(
    filtered_price_df["Price_PKR"],
    bins=40,
    kde=True,
    color=COLORS["secondary"],
    alpha=0.75,
    edgecolor="white",
    linewidth=0.4,
    line_kws={"linewidth":2},
    ax=ax
)

ax.set_xlabel("Price (PKR)")
ax.set_ylabel("Frequency")

ax.xaxis.set_major_formatter(
    mtick.FuncFormatter(format_millions)
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/price_distribution_filtered.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ------------------------------------------------------------
# Figure 4 : Mileage vs Price
# Purpose:
# Illustrate the overall depreciation trend after removing
# only the most extreme 1% of mileage and price values.
# ------------------------------------------------------------

filtered_df2 = df[
    (df["Mileage_KM"] < df["Mileage_KM"].quantile(0.99))
    &
    (df["Price_PKR"] < df["Price_PKR"].quantile(0.99))
]

fig, ax = plt.subplots(figsize=(8,5))

sns.regplot(
    data=filtered_df2,
    x="Mileage_KM",
    y="Price_PKR",
    
    scatter_kws = {
        "s": 12,
        "alpha": 0.20,
        "color": COLORS["accent"]
    },

    line_kws = {
        "linewidth": 2.4,
        "color": "black"
    },

    ci=None,
    ax=ax
)

ax.set_xlabel("Mileage (KM)")
ax.set_ylabel("Price (PKR)")

ax.set_ylim(bottom=0)

ax.yaxis.set_major_formatter(
    mtick.FuncFormatter(format_millions)
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/mileage_vs_price_filtered.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

print("Additional exploratory figures generated successfully.")

In [ ]:
# ============================================================
# Figure : Price Distribution by City
# ============================================================

fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="City",
    y="Price_PKR",
    color=COLORS["primary"],
    width=0.55,
    linewidth=1.2,
    showmeans=True,
    meanprops={
        "marker": "D",
        "markerfacecolor": "white",
        "markeredgecolor": "black",
        "markersize": 5
    },
    ax=ax
)

ax.set_xlabel("City")
ax.set_ylabel("Price (PKR)")

ax.yaxis.set_major_formatter(
    mtick.FuncFormatter(format_millions)
)

style_axes(ax)
from matplotlib.lines import Line2D

mean_legend = Line2D(
    [0], [0],
    marker='D',
    color='none',
    markerfacecolor='white',
    markeredgecolor='black',
    markersize=6,
    label='Mean'
)

ax.legend(
    handles=[mean_legend],
    loc='upper right',
    frameon=True,
    fontsize=10
)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/price_by_city.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure : Distribution of Cars by Manufacturing Year
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

sns.countplot(
    data=df,
    x="Year",
    order=sorted(df["Year"].dropna().unique()),
    color=COLORS["secondary"],
    edgecolor="white",
    linewidth=0.4,
    width=0.8,
    ax=ax
)

ax.set_xlabel("Manufacturing Year")
ax.set_ylabel("Number of Listings")

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter('{x:,.0f}')
)

plt.xticks(
    rotation=45,
    ha="right"
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/year_distribution.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure : Distribution of Engine Capacity
# ============================================================

from matplotlib.ticker import MultipleLocator

fig, ax = plt.subplots(figsize=(8, 5))

sns.histplot(
    df["Engine_CC"],
    bins=30,
    kde=True,
    color=COLORS["primary"],
    alpha=0.75,
    edgecolor="white",
    linewidth=0.4,
    line_kws={"linewidth": 2},
    ax=ax
)

ax.set_xlabel("Engine Capacity (CC)")
ax.set_ylabel("Number of Listings")

ax.xaxis.set_major_locator(MultipleLocator(500))

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter('{x:,.0f}')
)

style_axes(ax)

plt.tight_layout()

plt.savefig(
    "report_feature_figures/engine_cc_distribution.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure : Number of Listings by City
# Purpose:
# Show the geographical distribution of vehicle listings.
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

city_order = df["City"].value_counts().index

sns.countplot(
    data=df,
    x="City",
    order=city_order,
    color=COLORS["secondary"],
    edgecolor="white",
    linewidth=0.4,
    width=0.50,
    ax=ax
)

ax.set_xlabel("City")
ax.set_ylabel("Number of Listings")

# Format y-axis with comma separators
ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter('{x:,.0f}')
)

style_axes(ax)

# ------------------------------------------------------------
# Display listing count above each bar
# ------------------------------------------------------------
for patch in ax.patches:
    height = patch.get_height()

    ax.annotate(
        f"{int(height):,}",
        (
            patch.get_x() + patch.get_width() / 2,
            height
        ),
        ha="center",
        va="bottom",
        fontsize=10,
        xytext=(0, 5),
        textcoords="offset points"
    )

plt.tight_layout()

plt.savefig(
    "report_feature_figures/city_distribution.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure: Fuel Type Distribution
# Categories contributing <1% are grouped into "Others"
# ============================================================

fig, ax = plt.subplots(figsize=(7, 6))

# Fuel type counts
fuel_counts = df["Fuel_Type"].value_counts()

# Group categories contributing less than 1%
threshold = len(df) * 0.01

major = fuel_counts[fuel_counts >= threshold]
others = fuel_counts[fuel_counts < threshold].sum()

if others > 0:
    fuel_counts = pd.concat([major, pd.Series({"Others": others})])
else:
    fuel_counts = major

colors = [
    COLORS["primary"],
    COLORS["secondary"],
    COLORS["accent"],
    "#BFC5CD"
]

# Display percentage labels only for slices >= 5%
def autopct_fmt(pct):
    return f"{pct:.1f}%" if pct >= 5 else ""

wedges, texts, autotexts = ax.pie(
    fuel_counts.values,
    labels=None,
    colors=colors[:len(fuel_counts)],
    startangle=90,
    counterclock=False,
    radius=1.08,
    wedgeprops=dict(
        width=0.30,
        edgecolor="white",
        linewidth=1
    ),
    autopct=autopct_fmt,
    pctdistance=0.80,
    textprops={
        "fontsize":10,
        "fontweight":"bold"
    }
)

# Improve visibility of percentage labels
for t in autotexts:
    t.set_color("white")
    t.set_path_effects([
        pe.withStroke(linewidth=2.2, foreground="#1F2937")
    ])

legend_labels = [
    f"{label} ({value/len(df)*100:.1f}%)"
    for label, value in fuel_counts.items()
]

ax.legend(
    wedges,
    legend_labels,
    title="Fuel Type",
    title_fontsize=11,
    fontsize=10,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

ax.set(aspect="equal")

plt.tight_layout()

plt.savefig(
    "report_feature_figures/fuel_type_distribution.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure: Transmission Distribution
# ============================================================

fig, ax = plt.subplots(figsize=(7, 6))

# Transmission counts
transmission_counts = df["Transmission"].value_counts()

colors = [
    COLORS["primary"],
    COLORS["secondary"]
]

wedges, texts, autotexts = ax.pie(
    transmission_counts.values,
    labels=None,
    colors=colors[:len(transmission_counts)],
    startangle=90,
    counterclock=False,
    radius=1.08,
    wedgeprops=dict(
        width=0.30,
        edgecolor="white",
        linewidth=1
    ),
    autopct="%1.1f%%",
    pctdistance=0.80,
    textprops={
        "fontsize":10,
        "fontweight":"bold"
    }
)

# Improve visibility of percentage labels
for t in autotexts:
    t.set_color("white")
    t.set_path_effects([
        pe.withStroke(linewidth=2.2, foreground="#1F2937")
    ])

legend_labels = [
    f"{label} ({value/len(df)*100:.1f}%)"
    for label, value in transmission_counts.items()
]

ax.legend(
    wedges,
    legend_labels,
    title="Transmission",
    title_fontsize=11,
    fontsize=10,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

ax.set(aspect="equal")

plt.tight_layout()

plt.savefig(
    "report_feature_figures/transmission_distribution.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

## 4. Feature Engineering — Company / Model Extraction

Listing titles (e.g. `"Toyota Corolla 2018 for Sale"`) are parsed into separate `Company`
and `Model` columns, handling multi-word brand names (Mercedes Benz, Land Rover) and
multi-word model names (Wagon R, Land Cruiser, C-HR, etc.) via lookup lists.


In [ ]:
import pandas as pd
import sqlite3

print("Fetching Model-Specific Market Analytics...")

# 1. Connect to database and load the cleaned dataset
conn = sqlite3.connect('pakwheels_dataset.db')
df = pd.read_sql_query("SELECT * FROM cleaned_cars", conn)

# 2. Known multi-word company names
multi_word_companies = [
    "Mercedes Benz",
    "Land Rover",
    "Range Rover"
]

# 3. Known multi-word model names
multi_word_models = [
    "Land Cruiser",
    "Wagon R",
    "Oshan X7",

    "N One",
    "N WGN",
    "N Wgn",
    "N Box",
    "N Box Custom",

    "C Class",
    "E Class",
    "S Class",
    "GLA Class",
    "GLC Class",
    "GLE Class",

    "BR-V",
    "HR-V",
    "C-HR"
]


multi_word_companies = sorted(multi_word_companies, key=len, reverse=True)
multi_word_models = sorted(multi_word_models, key=len, reverse=True)


def extract_company_model(title):
    """
    Extract Company and Model from the car title.

    Example:
    Toyota Corolla 2018 for Sale  -> Company = Toyota, Model = Corolla
    Suzuki Wagon R 2021 for Sale  -> Company = Suzuki, Model = Wagon R
    Toyota Land Cruiser 2017      -> Company = Toyota, Model = Land Cruiser
    Mercedes Benz C Class 2018    -> Company = Mercedes Benz, Model = C Class
    """

    title = str(title).strip() # ensure the title is a string and remove extra spaces

    if title == "" or title.lower() == "nan":
        return pd.Series(["Unknown", "Unknown"])

    words = title.split()

    if len(words) == 0:
        return pd.Series(["Unknown", "Unknown"])

    title_lower = title.lower()

    # Step 1: Check multi-word company names first
    for company in multi_word_companies:
        company_lower = company.lower()

        if title_lower.startswith(company_lower):
            remaining_title = title[len(company):].strip() #remove the company name from the titlle
            remaining_words = remaining_title.split()

            if len(remaining_words) == 0:
                return pd.Series([company, "Unknown"])

            remaining_lower = remaining_title.lower()

            # Step 2: Check multi-word model after multi-word company
            for model in multi_word_models:
                model_lower = model.lower()

                if remaining_lower.startswith(model_lower):
                    return pd.Series([company, model])

            # Step 3: if its a single word model for a multi word company
            return pd.Series([company, remaining_words[0]])

    # Step 4: Default company = first word
    company = words[0]

    if len(words) == 1:
        return pd.Series([company, "Unknown"])

    remaining_title = " ".join(words[1:]) # removes the first word of the title and joins the rest back into a string
    remaining_lower = remaining_title.lower()

    # Step 5: Check multi-word model names
    for model in multi_word_models:
        model_lower = model.lower()

        if remaining_lower.startswith(model_lower):
            return pd.Series([company, model])

    # Step 6: Default model = second word
    model = words[1]

    return pd.Series([company, model])


# 4. Apply company/model extraction
df[['Company', 'Model']] = df['Title'].apply(extract_company_model)

# 5. Save engineered dataset for next features
# This table will be used for saleability prediction and missing attribute estimation.
df.to_sql('market_analysis_cars', conn, if_exists='replace', index=False)

# 6. Group by Company and Model for model-specific market analytics
model_stats = df.groupby(['Company', 'Model']).agg(
    Total_Listings=('Title', 'count'),
    Avg_Price_PKR=('Price_PKR', 'mean'),
    Median_Price_PKR=('Price_PKR', 'median'),
    Avg_Mileage_KM=('Mileage_KM', 'mean'),
    Median_Mileage_KM=('Mileage_KM', 'median')
).sort_values(by='Total_Listings', ascending=False).head(40)

# 7. Format values for readable output
model_stats['Avg_Price_PKR'] = model_stats['Avg_Price_PKR'].apply(lambda x: f"PKR {x:,.0f}")
model_stats['Median_Price_PKR'] = model_stats['Median_Price_PKR'].apply(lambda x: f"PKR {x:,.0f}")
model_stats['Avg_Mileage_KM'] = model_stats['Avg_Mileage_KM'].apply(lambda x: f"{x:,.0f} km")
model_stats['Median_Mileage_KM'] = model_stats['Median_Mileage_KM'].apply(lambda x: f"{x:,.0f} km")

print("\n--- TOP 40 SPECIFIC CAR MODELS IN THE MARKET ---")
print(model_stats.to_string())

conn.close()

## 5. Baseline Model — Decision Tree Regressor + Saleability Scoring

`Company`, `Model`, `City`, `Fuel_Type`, and `Transmission` are ordinal-encoded; `Year`,
`Mileage_KM`, and `Engine_CC` pass through unchanged. The tree is depth- and leaf-size-capped
to control overfitting (see the separate hyperparameter tuning notebook for how these values
were chosen). `predict_saleability()` compares an asking price against the model's predicted
market price to flag a listing as under/fairly/overpriced.


In [ ]:
import sqlite3
import pandas as pd


from sklearn.model_selection import train_test_split # This function splits the dataset into a training set and a testing set. The training set is used to train the model, while the testing set is used to evaluate the model's performance on unseen data.
from sklearn.tree import DecisionTreeRegressor # such tree is used when target value is numeric
from sklearn.compose import ColumnTransformer  # This is used to apply different preprocessing to different columns.
# categorical columns → OneHotEncoder
# numeric columns → pass directly without changes
from sklearn.preprocessing import OrdinalEncoder# This is used to convert categorical text data into a format that can be provided to machine learning algorithms to do a better job in prediction. It creates new binary columns for each category level.
# integer for each catagorical column


from sklearn.pipeline import Pipeline # This is used to chain multiple steps together into a single object. It allows us to combine preprocessing and modeling steps so that we can treat the entire process as a single unit. For example, we can create a pipeline that first applies the ColumnTransformer to preprocess the data and then fits a DecisionTreeRegressor to the preprocessed data.
from sklearn.metrics import mean_absolute_error, r2_score 
# mean_absolute_error tells the average prediction error in PKR.
# r2_score tells how much variation in price the model explain

print("Training Regression Tree for Price and Saleability Prediction...")

# 1. Load engineered dataset
conn = sqlite3.connect('pakwheels_dataset.db')
df = pd.read_sql_query("SELECT * FROM market_analysis_cars", conn)
conn.close()

# 2. Select useful columns for prediction
# These columns affect car price in the market.
required_columns = [
    'Company',
    'Model',
    'City',
    'Year',
    'Mileage_KM',
    'Engine_CC',
    'Fuel_Type',
    'Transmission',
    'Price_PKR'
]

df = df[required_columns].dropna()

#  Define input features and target
X = df.drop('Price_PKR', axis=1) #axis =1 means we are dropping a column, axis=0 would mean dropping a row
# For structural changes (drop), axis=1 means "Target the columns."
# For computations (apply), axis=1 means "Calculate horizontally (which processes one row at a time)."


y = df['Price_PKR']

# 5. Separate categorical and numeric columns
categorical_features = [
    'Company',
    'Model',
    'City',
    'Fuel_Type',
    'Transmission'
]

numeric_features = [
    'Year',
    'Mileage_KM',
    'Engine_CC'
]

# 6. Convert categorical columns into machine-readable format
preprocessor = ColumnTransformer(
    transformers=[
        # Name this transformation 'cat'
        # Use OneHotEncoder
        # Apply it to categorical_features
        ('cat',
            OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1
        ),
        categorical_features), # handle_unknown='ignore' means that if we encounter a category in the test set that was not present in the training set, we convert catagorical data into numerical matrix
        ('num', 'passthrough', numeric_features) # 'passthrough' means that we keep the numeric features as it is
    ]
)

# 7. Create Regression Tree model
# max_depth limits tree complexity.
# min_samples_leaf prevents the model from making decisions based on too few records.
regression_tree_model = DecisionTreeRegressor(
    max_depth=8,  # limits the tree to 8 decision levels to prevent overfitting
    min_samples_leaf=10, # final decisions will be based on at least 10 records to avoid overfitting
    random_state=42 
)

# 8. Create full ML pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', regression_tree_model)
])

# 9. Split data into training and testing parts
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 10. Train the Regression Tree
model.fit(X_train, y_train)

# 11. Test model performance
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\n--- REGRESSION TREE MODEL PERFORMANCE ---")
print(f"Training Records: {len(X_train)}")
print(f"Testing Records: {len(X_test)}")
print(f"Mean Absolute Error: PKR {mae:,.0f}")
print(f"R-Squared Score: {r2:.2f}")


# 12. Function to calculate saleability
def predict_saleability(company, model_name, city, year, mileage, engine_cc, fuel_type, transmission, asking_price):


    # Create user input as DataFrame because pipeline was designed for dataframe
    user_car = pd.DataFrame([{
        'Company': company,
        'Model': model_name,
        'City': city,
        'Year': year,
        'Mileage_KM': mileage,
        'Engine_CC': engine_cc,
        'Fuel_Type': fuel_type,
        'Transmission': transmission
    }])

    # Predict market price
    predicted_price = model.predict(user_car)[0] #[0] because predict returns an array, we want the first element which is the predicted price

    # Calculate price difference percentage
    price_difference = asking_price - predicted_price
    price_difference_percent = (price_difference / predicted_price) * 100

    # Saleability decision based on asking price vs predicted market price
    if price_difference_percent < -15:
        saleability = "Very High Saleability / Underpriced"
        reason = (
            "The asking price is far below the predicted market price. "
            "It may sell quickly, but condition, documents, accident history, and listing accuracy should be checked."
        )
    elif -15 <= price_difference_percent < -5:
        saleability = "High Saleability / Slightly Under Market"
        reason = "The asking price is below the predicted market price, so the car is likely attractive to buyers."
    elif -5 <= price_difference_percent <= 5:
        saleability = "High Saleability / Fair Market Price"
        reason = "The asking price is close to the predicted market price."
    elif 5 < price_difference_percent <= 15:
        saleability = "Medium Saleability"
        reason = "The asking price is slightly above the predicted market price."
    else:
        saleability = "Low Saleability"
        reason = "The asking price is much higher than the predicted market price."


    print("\n--- SALEABILITY PREDICTION RESULT ---")
    print(f"Company: {company}")
    print(f"Model: {model_name}")
    print(f"City: {city}")
    print(f"Year: {year}")
    print(f"Mileage: {mileage:,} km")
    print(f"Engine: {engine_cc} cc")
    print(f"Fuel Type: {fuel_type}")
    print(f"Transmission: {transmission}")
    print(f"Asking Price: PKR {asking_price:,.0f}")
    print(f"Predicted Market Price: PKR {predicted_price:,.0f}")
    print(f"Price Difference: PKR {price_difference:,.0f}")
    print(f"Price Difference Percentage: {price_difference_percent:.2f}%")
    print(f"Saleability: {saleability}")
    print(f"Reason: {reason}")


# 13. Example test case
# You can change these values according to the car you want to test.
predict_saleability(
    company="Toyota",
    model_name="Prado",
    city="Islamabad",
    year=2020,
    mileage=60000,
    engine_cc=2700,
    fuel_type="Petrol",
    transmission="manual",
    asking_price=8000000
)

## 6. Model Evaluation

Error breakdown on the held-out test set, plus performance split by price segment
(low-budget / mid-range / high-value / luxury) since a flat MAE hides how error scales
with price.


In [ ]:
results = X_test.copy().reset_index(drop=True)

results["Actual_Price"] = y_test.reset_index(drop=True)
results["Predicted_Price"] = predictions
results["Absolute_Error"] = abs(
    results["Actual_Price"] - results["Predicted_Price"]
)
results["Percentage_Error"] = (
    results["Absolute_Error"] / results["Actual_Price"]
) * 100

print(results.sort_values("Absolute_Error", ascending=False).head(20))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Create folder for saving result figures
os.makedirs("result_evaluation_figures", exist_ok=True)

# Convert y_test and predictions into clean arrays
actual_prices = np.array(y_test).astype(float)
predicted_prices = np.array(predictions).astype(float)

# Create evaluation dataframe
eval_df = pd.DataFrame({
    "Actual_Price": actual_prices,
    "Predicted_Price": predicted_prices
})

# Absolute Error
eval_df["Absolute_Error"] = abs(eval_df["Actual_Price"] - eval_df["Predicted_Price"])

# Percentage Error
eval_df["Percentage_Error"] = (eval_df["Absolute_Error"] / eval_df["Actual_Price"]) * 100

# Overall MAPE
mape = eval_df["Percentage_Error"].mean()

print("Decision Tree Evaluation")
print("------------------------")
print(f"MAE  : PKR {mae:,.0f}")
print(f"R²   : {r2:.2f}")
print(f"MAPE : {mape:.2f}%")

In [ ]:
# Define price segments based on actual car price
bins = [0, 2_000_000, 5_000_000, 10_000_000, np.inf]

labels = [
    "Low-budget (< 2M)",
    "Mid-range (2M - 5M)",
    "High-value (5M - 10M)",
    "Luxury/SUV (> 10M)"
]

eval_df["Price_Segment"] = pd.cut(
    eval_df["Actual_Price"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Segment-wise evaluation
segment_summary = eval_df.groupby("Price_Segment").agg(
    Test_Records=("Actual_Price", "count"),
    Avg_Actual_Price=("Actual_Price", "mean"),
    MAE=("Absolute_Error", "mean"),
    MAPE=("Percentage_Error", "mean")
).reset_index()

# Round values
segment_summary["Avg_Actual_Price"] = segment_summary["Avg_Actual_Price"].round(0)
segment_summary["MAE"] = segment_summary["MAE"].round(0)
segment_summary["MAPE"] = segment_summary["MAPE"].round(2)

print("\nSegment-wise Error Summary")
print("--------------------------")
print(segment_summary)

# Save CSV backup
segment_summary.to_csv("segment_wise_error_summary.csv", index=False)

# Create a formatted version for easier report writing
segment_summary_formatted = segment_summary.copy()

segment_summary_formatted["Avg_Actual_Price"] = segment_summary_formatted["Avg_Actual_Price"].apply(
    lambda x: f"PKR {x:,.0f}" if pd.notnull(x) else "-"
)

segment_summary_formatted["MAE"] = segment_summary_formatted["MAE"].apply(
    lambda x: f"PKR {x:,.0f}" if pd.notnull(x) else "-"
)

segment_summary_formatted["MAPE"] = segment_summary_formatted["MAPE"].apply(
    lambda x: f"{x:.2f}%" if pd.notnull(x) else "-"
)

print("\nFormatted Segment-wise Table")
print("----------------------------")
print(segment_summary_formatted)

In [ ]:
# ============================================================
# Figure: Actual vs Predicted Vehicle Prices
# ============================================================

fig, ax = plt.subplots(figsize=(7.5, 6))

# Scatter plot
ax.scatter(
    eval_df["Actual_Price"],
    eval_df["Predicted_Price"],
    s=28,
    alpha=0.45,
    color=COLORS["primary"],
    edgecolors="none"
)

# Identity (Perfect Prediction) Line
min_price = min(
    eval_df["Actual_Price"].min(),
    eval_df["Predicted_Price"].min()
)

max_price = max(
    eval_df["Actual_Price"].max(),
    eval_df["Predicted_Price"].max()
)

ax.plot(
    [min_price, max_price],
    [min_price, max_price],
    linestyle="--",
    linewidth=2,
    color=COLORS["accent"],
    label="Perfect Prediction"
)

# Labels
ax.set_xlabel("Actual Price (PKR)")
ax.set_ylabel("Predicted Price (PKR)")

# Format axis values
ax.xaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

# Apply notebook-wide styling (Cell 15)
style_axes(ax)

# Equal aspect ratio
ax.set_aspect("equal", adjustable="box")

# Performance summary
metrics_text = (
    f"R² = {r2:.2f}\n"
    f"MAE = {mae:,.0f} PKR\n"
    f"MAPE = {mape:.2f}%"
)

ax.text(
    0.03,
    0.97,
    metrics_text,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox=dict(
        facecolor="white",
        edgecolor="lightgray",
        boxstyle="round,pad=0.4"
    )
)

# Legend
ax.legend(frameon=False, loc="lower right", bbox_to_anchor=(1.0, 0.05))

plt.tight_layout()

plt.savefig(
    "result_evaluation_figures/actual_vs_predicted_price.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure: Segment-wise Mean Absolute Error (MAE)
# ============================================================

label_map = {
    "Low-budget (< 2M)": "Low\n(<2M)",
    "Mid-range (2M - 5M)": "Mid\n(2–5M)",
    "High-value (5M - 10M)": "High\n(5–10M)",
    "Luxury/SUV (> 10M)": "Luxury\n(>10M)",
}

labels = segment_summary["Price_Segment"].astype(str).map(label_map)

fig, ax = plt.subplots(figsize=(6.8, 5.5))

bars = ax.bar(
    labels,
    segment_summary["MAE"],
    width=0.55,
    color=COLORS["primary"],
)

# Value labels
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:,.0f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_xlabel("Price Segment")
ax.set_ylabel("Mean Absolute Error (PKR)")

# Format y-axis
ax.yaxis.set_major_formatter(
    mtick.StrMethodFormatter("{x:,.0f}")
)

# Apply notebook styling
style_axes(ax)

plt.tight_layout()

plt.savefig(
    "result_evaluation_figures/segment_wise_mae.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

In [ ]:
# ============================================================
# Figure: Segment-wise Mean Absolute Percentage Error (MAPE)
# ============================================================

label_map = {
    "Low-budget (< 2M)": "Low\n(<2M)",
    "Mid-range (2M - 5M)": "Mid\n(2–5M)",
    "High-value (5M - 10M)": "High\n(5–10M)",
    "Luxury/SUV (> 10M)": "Luxury\n(>10M)",
}

labels = segment_summary["Price_Segment"].astype(str).map(label_map)

# Calculate MAPE for each price segment
segment_mape = (
    eval_df.groupby("Price_Segment", observed=True)["Percentage_Error"]
    .mean()
    .reset_index()
)

segment_mape["Price_Segment"] = (
    segment_mape["Price_Segment"]
    .astype(str)
    .map(label_map)
)

fig, ax = plt.subplots(figsize=(6.8, 5.5))

bars = ax.bar(
    segment_mape["Price_Segment"],
    segment_mape["Percentage_Error"],
    width=0.55,
    color=COLORS["primary"]
)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.3,
        f"{height:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )


ax.set_xlabel("Price Segment")
ax.set_ylabel("MAPE (%)")

ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()

plt.savefig(
    "result_evaluation_figures/segment_wise_mape.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.05
)

plt.show()

## 7. KNN-Based Missing Attribute Estimation

A separate, self-contained use case: given *partial* information about a car (e.g. only
company + model, or only price + engine size), find the most similar listings via
K-Nearest-Neighbors and use them to estimate the missing fields (year, mileage, price, etc.),
plus a demand-level read on how common that car is in the dataset.


In [ ]:
import sqlite3

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import time
from sklearn.neighbors import NearestNeighbors
import pandas as pd


print("Running Flexible Missing Attribute Estimation Using KNN...")

# 1. Load engineered dataset
conn = sqlite3.connect('pakwheels_dataset.db')
df = pd.read_sql_query("SELECT * FROM market_analysis_cars", conn)
conn.close()

# 2. Required columns
required_columns = [
    'Company',
    'Model',
    'City',
    'Year',
    'Mileage_KM',
    'Engine_CC',
    'Fuel_Type',
    'Transmission',
    'Price_PKR'
]

df = df[required_columns].dropna()

# 3. Convert numeric columns
df['Year'] = df['Year'].astype(int)
df['Mileage_KM'] = df['Mileage_KM'].astype(int)
df['Engine_CC'] = df['Engine_CC'].astype(int)
df['Price_PKR'] = df['Price_PKR'].astype(int)

categorical_columns = [
    'Company',
    'Model',
    'City',
    'Fuel_Type',
    'Transmission'
]

numeric_columns = [
    'Year',
    'Mileage_KM',
    'Engine_CC',
    'Price_PKR'
]


def match_existing_value(value, column_name):
    """
    Matches user input with existing dataset values.
    Example:
    'toyota' -> 'Toyota'
    'corolla' -> 'Corolla'
    """
    if value is None:
        return None

    value = str(value).strip()

    unique_values = df[column_name].dropna().astype(str).unique()

    for existing_value in unique_values:
        if value.lower() == existing_value.lower():
            return existing_value

    return value


def mode_value(series):
    """
    Returns the most common value from a column.
    """
    mode_result = series.mode()

    if len(mode_result) == 0:
        return "Unknown"

    return mode_result.iloc[0]


def demand_label(count):
    """
    Converts number of market records into demand level.
    """
    if count >= 75:
        return "High Demand"
    elif count >= 25:
        return "Medium Demand"
    else:
        return "Low Demand"


def print_known_inputs(user_inputs):
    print("\n--- USER PROVIDED INFORMATION ---")

    for feature, value in user_inputs.items():
        if value is not None:
            if feature == 'Price_PKR':
                print(f"{feature}: PKR {value:,.0f}")
            elif feature == 'Mileage_KM':
                print(f"{feature}: {value:,} km")
            elif feature == 'Engine_CC':
                print(f"{feature}: {value} cc")
            else:
                print(f"{feature}: {value}")


def estimate_from_profile(candidates, user_inputs):
    """
    Used when the user provides very little information.
    This gives market profile, not strict KNN.
    """

    print("\n--- MODE USED: MARKET PROFILE ---")
    print("Reason: Only limited information was provided, so exact KNN comparison is weak.")
    print("The system is estimating missing values from matching market records.")

    total_records = len(candidates)

    estimated_values = {
        'Company': user_inputs['Company'] if user_inputs['Company'] is not None else mode_value(candidates['Company']),
        'Model': user_inputs['Model'] if user_inputs['Model'] is not None else mode_value(candidates['Model']),
        'City': user_inputs['City'] if user_inputs['City'] is not None else mode_value(candidates['City']),
        'Year': user_inputs['Year'] if user_inputs['Year'] is not None else int(round(candidates['Year'].median())),
        'Mileage_KM': user_inputs['Mileage_KM'] if user_inputs['Mileage_KM'] is not None else int(round(candidates['Mileage_KM'].median())),
        'Engine_CC': user_inputs['Engine_CC'] if user_inputs['Engine_CC'] is not None else int(round(candidates['Engine_CC'].median())),
        'Fuel_Type': user_inputs['Fuel_Type'] if user_inputs['Fuel_Type'] is not None else mode_value(candidates['Fuel_Type']),
        'Transmission': user_inputs['Transmission'] if user_inputs['Transmission'] is not None else mode_value(candidates['Transmission']),
        'Price_PKR': user_inputs['Price_PKR'] if user_inputs['Price_PKR'] is not None else int(round(candidates['Price_PKR'].median()))
    }

    price_q1 = int(round(candidates['Price_PKR'].quantile(0.25)))
    price_q3 = int(round(candidates['Price_PKR'].quantile(0.75)))

    mileage_q1 = int(round(candidates['Mileage_KM'].quantile(0.25)))
    mileage_q3 = int(round(candidates['Mileage_KM'].quantile(0.75)))

    year_q1 = int(round(candidates['Year'].quantile(0.25)))
    year_q3 = int(round(candidates['Year'].quantile(0.75)))

    print("\n--- ESTIMATED MISSING ATTRIBUTES ---")

    for feature, value in estimated_values.items():
        if user_inputs[feature] is None:
            if feature == 'Price_PKR':
                print(f"Estimated {feature}: PKR {value:,.0f}")
            elif feature == 'Mileage_KM':
                print(f"Estimated {feature}: {value:,} km")
            elif feature == 'Engine_CC':
                print(f"Estimated {feature}: {value} cc")
            else:
                print(f"Estimated {feature}: {value}")

    print("\n--- MARKET RANGE INFORMATION ---")
    print(f"Matching Records Found: {total_records}")
    print(f"Demand Level: {demand_label(total_records)}")
    print(f"Typical Price Range: PKR {price_q1:,.0f} - PKR {price_q3:,.0f}")
    print(f"Typical Year Range: {year_q1} - {year_q3}")
    print(f"Typical Mileage Range: {mileage_q1:,} km - {mileage_q3:,} km")


def estimate_missing_attributes(
    company=None,
    model_name=None,
    city=None,
    year=None,
    mileage=None,
    engine_cc=None,
    fuel_type=None,
    transmission=None,
    price=None,
    k=10
):
    """
    Flexible Missing Attribute Estimation.

    User may provide any known fields.
    The system estimates the missing fields.

    If only very basic information is provided, it gives a market profile.
    If enough information is provided, it uses KNN to find similar cars.
    """

    # 1. Normalize categorical inputs
    company = match_existing_value(company, 'Company') if company is not None else None
    model_name = match_existing_value(model_name, 'Model') if model_name is not None else None
    city = match_existing_value(city, 'City') if city is not None else None
    fuel_type = match_existing_value(fuel_type, 'Fuel_Type') if fuel_type is not None else None
    transmission = match_existing_value(transmission, 'Transmission') if transmission is not None else None

    # 2. Store user inputs
    user_inputs = {
        'Company': company,
        'Model': model_name,
        'City': city,
        'Year': year,
        'Mileage_KM': mileage,
        'Engine_CC': engine_cc,
        'Fuel_Type': fuel_type,
        'Transmission': transmission,
        'Price_PKR': price
    }

    provided_features = [
        feature for feature, value in user_inputs.items()
        if value is not None
    ]

    if len(provided_features) == 0:
        print("Error: Please provide at least one known car detail.")
        return

    print_known_inputs(user_inputs)

    # 3. Narrow the candidate dataset using Company and Model if available
    # This makes the estimation more relevant.
    candidates = df.copy()

    if company is not None:
        candidates = candidates[candidates['Company'] == company]

    if model_name is not None:
        candidates = candidates[candidates['Model'] == model_name]

    if len(candidates) == 0:
        print("\nError: No matching records found for the provided Company/Model.")
        return

    # 4. Decide whether KNN is meaningful
    # Company and Model alone are not enough for strong KNN because all candidates are already the same type.
    extra_features = []

    if city is not None:
        extra_features.append('City')

    if year is not None:
        extra_features.append('Year')

    if mileage is not None:
        extra_features.append('Mileage_KM')

    if engine_cc is not None:
        extra_features.append('Engine_CC')

    if fuel_type is not None:
        extra_features.append('Fuel_Type')

    if transmission is not None:
        extra_features.append('Transmission')

    if price is not None:
        extra_features.append('Price_PKR')

    # 5. If user only gave Company/Model or very limited info, use profile estimation
    if len(extra_features) == 0:
        estimate_from_profile(candidates, user_inputs)
        return

    # 6. Prepare KNN using the extra known features
    working_df = candidates.dropna(subset=extra_features).copy()

    if len(working_df) == 0:
        print("\nError: No usable records found after applying provided features.")
        return

    provided_categorical = [
        feature for feature in extra_features
        if feature in categorical_columns
    ]

    provided_numeric = [
        feature for feature in extra_features
        if feature in numeric_columns
    ]

    transformers = []

    if len(provided_categorical) > 0:
        try:
            encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        except TypeError:
            encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

        transformers.append(('cat', encoder, provided_categorical))

    if len(provided_numeric) > 0:
        transformers.append(('num', StandardScaler(), provided_numeric))
    ############################################
    preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder='drop'
    )

    user_row = pd.DataFrame([{
        feature: user_inputs[feature]
        for feature in extra_features
    }])

    total_start = time.perf_counter()

    preprocessing_start = time.perf_counter()

    X_dataset = preprocessor.fit_transform(working_df[extra_features])
    X_user = preprocessor.transform(user_row[extra_features])

    preprocessing_end = time.perf_counter()
    ######################################
    k_actual = min(k, len(working_df))

    knn = NearestNeighbors(
        n_neighbors=k_actual,
        metric='euclidean'
    )
    ################# timer #########################
    fit_start = time.perf_counter()

    knn.fit(X_dataset)

    fit_end = time.perf_counter()
    ############### timer ##########################
    
    ##################################
    prediction_start = time.perf_counter()
    distances, indices = knn.kneighbors(X_user)
    prediction_end = time.perf_counter()
    total_end = time.perf_counter()
    ##################################F

    similar_cars = working_df.iloc[indices[0]].copy()
    similar_cars['KNN_Distance'] = distances[0]
    similar_cars['Similarity_Rank'] = range(1, len(similar_cars) + 1)

    # 7. Estimate only the missing values
    estimated_values = {
        'Company': company if company is not None else mode_value(similar_cars['Company']),
        'Model': model_name if model_name is not None else mode_value(similar_cars['Model']),
        'City': city if city is not None else mode_value(similar_cars['City']),
        'Year': year if year is not None else int(round(similar_cars['Year'].median())),
        'Mileage_KM': mileage if mileage is not None else int(round(similar_cars['Mileage_KM'].median())),
        'Engine_CC': engine_cc if engine_cc is not None else int(round(similar_cars['Engine_CC'].median())),
        'Fuel_Type': fuel_type if fuel_type is not None else mode_value(similar_cars['Fuel_Type']),
        'Transmission': transmission if transmission is not None else mode_value(similar_cars['Transmission']),
        'Price_PKR': price if price is not None else int(round(similar_cars['Price_PKR'].median()))
    }

    price_q1 = int(round(similar_cars['Price_PKR'].quantile(0.25)))
    price_q3 = int(round(similar_cars['Price_PKR'].quantile(0.75)))

    mileage_q1 = int(round(similar_cars['Mileage_KM'].quantile(0.25)))
    mileage_q3 = int(round(similar_cars['Mileage_KM'].quantile(0.75)))
    
    
    print("\n--- COMPUTATIONAL PERFORMANCE ---")

    print(f"Feature Dimensions : {X_dataset.shape[1]}")
    print(f"Preprocessing Time : {preprocessing_end - preprocessing_start:.6f} sec")
    print(f"KNN Fit Time       : {fit_end - fit_start:.6f} sec")
    print(f"Prediction Time    : {prediction_end - prediction_start:.6f} sec")
    print(f"Total Time         : {total_end - total_start:.6f} sec")
    print("\n--- MODE USED: KNN NEAREST SIMILAR CARS ---")
    print(f"KNN was applied using these known features: {extra_features}")
    print(f"Nearest similar cars used: {k_actual}")

    print("\n--- ESTIMATED MISSING ATTRIBUTES ---")

    for feature, value in estimated_values.items():
        if user_inputs[feature] is None:
            if feature == 'Price_PKR':
                print(f"Estimated {feature}: PKR {value:,.0f}")
            elif feature == 'Mileage_KM':
                print(f"Estimated {feature}: {value:,} km")
            elif feature == 'Engine_CC':
                print(f"Estimated {feature}: {value} cc")
            else:
                print(f"Estimated {feature}: {value}")

    print("\n--- RANGE AMONG SIMILAR CARS ---")
    print(f"Typical Price Range: PKR {price_q1:,.0f} - PKR {price_q3:,.0f}")
    print(f"Typical Mileage Range: {mileage_q1:,} km - {mileage_q3:,} km")

    if company is not None and model_name is not None:
        market_count = len(df[(df['Company'] == company) & (df['Model'] == model_name)])
    else:
        market_count = len(candidates)

    print(f"Matching Market Records: {market_count}")
    print(f"Demand Level: {demand_label(market_count)}")

    # 8. Show nearest cars used by KNN
    print(f"\n--- TOP {k_actual} SIMILAR CARS USED BY KNN ---")

    display_columns = [
        'Similarity_Rank',
        'Company',
        'Model',
        'City',
        'Year',
        'Mileage_KM',
        'Engine_CC',
        'Fuel_Type',
        'Transmission',
        'Price_PKR',
        'KNN_Distance'
    ]

    similar_output = similar_cars[display_columns].copy()

    similar_output['Mileage_KM'] = similar_output['Mileage_KM'].apply(lambda x: f"{x:,.0f} km")
    similar_output['Engine_CC'] = similar_output['Engine_CC'].apply(lambda x: f"{x:,.0f} cc")
    similar_output['Price_PKR'] = similar_output['Price_PKR'].apply(lambda x: f"PKR {x:,.0f}")
    similar_output['KNN_Distance'] = similar_output['KNN_Distance'].apply(lambda x: f"{x:.2f}")

    print(similar_output.to_string(index=False))


# -----------------------------
# TEST EXAMPLES
# -----------------------------

# # Example 1:
# # User only knows company and model.
# # This will use market profile, not strict KNN.
estimate_missing_attributes(
    company="Suzuki",
    model_name="Alto"
)

# Example 2:
# User knows more details.
# This will use KNN.
# Uncomment to test.

# estimate_missing_attributes(
#     company="Toyota",
#     model_name="Corolla",
#     year=2018,
#     mileage=90000,
#     k=10
# )

# Example 3:
# User does not know company/model but knows price and engine.
# This will search broader market patterns.
# Uncomment to test.

estimate_missing_attributes(
    engine_cc=1300,
    price=4000000,
    transmission="Automatic",
    k=10
)

In [ ]:
# ============================================================
# PREPROCESSING RESULTS FOR REPORT
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import pandas as pd

# Use the same dataset used for KNN
preprocessing_df = df.copy()

# Build preprocessing pipeline
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_columns),
        ('num', StandardScaler(), numeric_columns)
    ]
)

X_transformed = preprocessor.fit_transform(
    preprocessing_df[categorical_columns + numeric_columns]
)

# ============================================================
# TABLE A
# Dataset dimensions before and after encoding
# ============================================================

print("\n--- DATASET DIMENSIONS ---")
print(f"Rows: {len(preprocessing_df)}")
print(f"Features before encoding: {len(categorical_columns) + len(numeric_columns)}")
print(f"Features after encoding: {X_transformed.shape[1]}")

# ============================================================
# TABLE B
# Scaling statistics
# ============================================================

scaler = StandardScaler()
scaled_numeric = scaler.fit_transform(preprocessing_df[numeric_columns])

scaling_results = pd.DataFrame({
    "Feature": numeric_columns,
    "Mean_After_Scaling": scaled_numeric.mean(axis=0),
    "Std_After_Scaling": scaled_numeric.std(axis=0)
})

print("\n--- SCALING RESULTS ---")
print(scaling_results)

# ============================================================
# TABLE C
# Encoding summary
# ============================================================

fitted_encoder = preprocessor.named_transformers_['cat']

encoded_columns = fitted_encoder.get_feature_names_out(
    categorical_columns
)

encoding_summary = pd.DataFrame({
    "Categorical Feature": categorical_columns,
    "Unique Categories": [
        preprocessing_df[col].nunique()
        for col in categorical_columns
    ]
})

print("\n--- ENCODING SUMMARY ---")
print(encoding_summary)

print("\n--- FIRST 20 ENCODED FEATURES ---")
for col in encoded_columns[:20]:
    print(col)